# Infer missing ages

This notebook finds defendant records where exactly one of `age_at_offence` or `age_at_sentencing` is present and both `judgement.charges.offence_date` and `judgement.judgment_date_time` are available.

It infers the missing age conservatively as either an exact age or an age range, writes those inferred values back to `verified-features`, and exports the `both_missing_cases` table to Excel.

The default query filters out `exclude=True` records to match the active verified dataset used elsewhere in the repo.

In [12]:
import calendar
import math
import os
from datetime import date, datetime, timedelta
from pathlib import Path

import pandas as pd
from bson import ObjectId
from dotenv import load_dotenv
from pymongo import MongoClient, UpdateOne

for dotenv_path in (
    Path.cwd() / '.env',
    Path.cwd() / 'notebooks' / '.env',
    Path.cwd().parent / 'notebooks' / '.env',
):
    if dotenv_path.exists():
        load_dotenv(dotenv_path)
        break

uri = os.getenv('DB_MONGODB_URI')
if not uri:
    raise RuntimeError('DB_MONGODB_URI is not set')

client = MongoClient(uri)
db = client.get_database(os.getenv('DB_NAME', 'drug-sentencing-predictor'))
verified_features = db.get_collection('verified-features')

pd.set_option('display.max_colwidth', None)

active_only = True


def parse_iso_date(value):
    if value is None or value == '':
        return None
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    if isinstance(value, str):
        return datetime.fromisoformat(value.replace('Z', '+00:00')).date()
    raise TypeError(f'Unsupported date value: {type(value)!r}')


def parse_date_interval(value):
    if value is None:
        return None
    if isinstance(value, dict):
        value = value.get('date')
    if isinstance(value, list):
        dates = [parse_iso_date(item) for item in value if item is not None]
        dates = [item for item in dates if item is not None]
        if not dates:
            return None
        return min(dates), max(dates)
    if pd.isna(value):
        return None
    parsed = parse_iso_date(value)
    if parsed is None:
        return None
    return parsed, parsed


def normalize_age_interval(value):
    if value is None:
        return None
    if isinstance(value, list):
        cleaned = [item for item in value if item is not None and not pd.isna(item)]
        if not cleaned:
            return None
        if len(cleaned) != 2:
            raise ValueError(f'Expected a 2-item age range, got {value!r}')
        low, high = sorted(int(item) for item in cleaned)
        return low, high
    if isinstance(value, float) and math.isnan(value):
        return None
    if pd.isna(value):
        return None
    return int(value), int(value)


def format_age_output(value):
    if value is None:
        return None
    low, high = value
    return low if low == high else [low, high]


def date_interval_text(interval):
    if interval is None:
        return None
    start, end = interval
    if start == end:
        return start.isoformat()
    return f'{start.isoformat()} to {end.isoformat()}'


def age_interval_text(interval):
    if interval is None:
        return None
    start, end = interval
    if start == end:
        return str(start)
    return f'{start} to {end}'


def build_age_record(age_value, source_text):
    return {
        'age': age_value,
        'source': source_text,
    }


def add_years(value, years):
    target_year = value.year + years
    target_day = min(value.day, calendar.monthrange(target_year, value.month)[1])
    return date(target_year, value.month, target_day)


def age_on_date(birth_date, target_date):
    years = target_date.year - birth_date.year
    if (target_date.month, target_date.day) < (birth_date.month, birth_date.day):
        years -= 1
    return years


def combine_intervals(intervals):
    valid_intervals = [interval for interval in intervals if interval is not None]
    if not valid_intervals:
        return None
    starts = [interval[0] for interval in valid_intervals]
    ends = [interval[1] for interval in valid_intervals]
    return min(starts), max(ends)


def infer_age_interval(source_age_interval, source_date_interval, target_date_interval):
    source_age_low, source_age_high = source_age_interval
    source_date_start, source_date_end = source_date_interval
    target_date_start, target_date_end = target_date_interval

    birth_earliest = add_years(source_date_start, -(source_age_high + 1)) + timedelta(days=1)
    birth_latest = add_years(source_date_end, -source_age_low)

    age_min = age_on_date(birth_latest, target_date_start)
    age_max = age_on_date(birth_earliest, target_date_end)

    if age_min > age_max:
        age_min, age_max = age_max, age_min

    return age_min, age_max


def to_object_id(value):
    if isinstance(value, ObjectId):
        return value
    return ObjectId(str(value))


In [13]:
pipeline = []
if active_only:
    pipeline.append({'$match': {'exclude': {'$ne': True}}})

pipeline.extend([
    {'$match': {'judgement.judgment_date_time': {'$ne': None}}},
    {'$unwind': '$defendants.defendants'},
    {'$unwind': '$judgement.charges'},
    {'$unwind': '$judgement.charges.defendants_of_charge'},
    {'$match': {'judgement.charges.offence_date': {'$ne': None}}},
    {
        '$match': {
            '$expr': {
                '$eq': [
                    '$defendants.defendants.defendant_id',
                    '$judgement.charges.defendants_of_charge.defendant_id',
                ]
            }
        }
    },
    {
        '$project': {
            'source_judgement_id': '$_id',
            'neutral_citation': '$judgement.neutral_citation',
            'defendant_id': '$defendants.defendants.defendant_id',
            'defendant_name': '$defendants.defendants.defendant_name.name',
            'charge_no': '$judgement.charges.charge_no',
            'offence_date': '$judgement.charges.offence_date.date',
            'judgment_date_time': '$judgement.judgment_date_time',
            'age_at_offence': '$defendants.defendants.age_at_offence.age',
            'age_at_sentencing': '$defendants.defendants.age_at_sentencing.age',
        }
    },
])

rows = list(verified_features.aggregate(pipeline, allowDiskUse=True))
raw_df = pd.DataFrame(rows)

if raw_df.empty:
    raise RuntimeError('No matching records were found')

raw_df['offence_date_interval'] = raw_df['offence_date'].apply(parse_date_interval)
raw_df['judgment_date'] = raw_df['judgment_date_time'].apply(parse_iso_date)
raw_df['age_at_offence_interval'] = raw_df['age_at_offence'].apply(normalize_age_interval)
raw_df['age_at_sentencing_interval'] = raw_df['age_at_sentencing'].apply(normalize_age_interval)

grouped_df = (
    raw_df.groupby(
        ['source_judgement_id', 'neutral_citation', 'defendant_id', 'defendant_name'],
        dropna=False,
        as_index=False,
    )
    .agg(
        charge_nos=('charge_no', lambda values: sorted({int(value) for value in values if pd.notna(value)})),
        offence_date_interval=('offence_date_interval', combine_intervals),
        judgment_date=('judgment_date', 'first'),
        age_at_offence_interval=('age_at_offence_interval', 'first'),
        age_at_sentencing_interval=('age_at_sentencing_interval', 'first'),
    )
    .sort_values(['neutral_citation', 'defendant_id'])
    .reset_index(drop=True)
)

def classify_row(row):
    if row['age_at_offence_interval'] is not None and row['age_at_sentencing_interval'] is None:
        return 'missing_age_at_sentencing'
    if row['age_at_offence_interval'] is None and row['age_at_sentencing_interval'] is not None:
        return 'missing_age_at_offence'
    if row['age_at_offence_interval'] is None and row['age_at_sentencing_interval'] is None:
        return 'both_missing'
    return 'complete'


grouped_df['age_status'] = grouped_df.apply(classify_row, axis=1)

candidate_rows = []
for _, row in grouped_df[grouped_df['age_status'].isin(['missing_age_at_offence', 'missing_age_at_sentencing'])].iterrows():
    if row['offence_date_interval'] is None or row['judgment_date'] is None:
        continue

    if row['age_status'] == 'missing_age_at_sentencing':
        source_age_interval = row['age_at_offence_interval']
        source_date_interval = row['offence_date_interval']
        target_date_interval = (row['judgment_date'], row['judgment_date'])
        inferred_interval = infer_age_interval(source_age_interval, source_date_interval, target_date_interval)
        candidate_rows.append({
            'source_judgement_id': row['source_judgement_id'],
            'neutral_citation': row['neutral_citation'],
            'defendant_id': row['defendant_id'],
            'defendant_name': row['defendant_name'],
            'charge_nos': row['charge_nos'],
            'judgment_date': row['judgment_date'].isoformat(),
            'offence_date_range': date_interval_text(row['offence_date_interval']),
            'age_at_offence': format_age_output(row['age_at_offence_interval']),
            'age_at_sentencing': format_age_output(row['age_at_sentencing_interval']),
            'missing_field': 'age_at_sentencing',
            'inferred_age_at_offence': None,
            'inferred_age_at_sentencing': format_age_output(inferred_interval),
            'inferred_age_range': age_interval_text(inferred_interval),
        })
    else:
        source_age_interval = row['age_at_sentencing_interval']
        source_date_interval = (row['judgment_date'], row['judgment_date'])
        target_date_interval = row['offence_date_interval']
        inferred_interval = infer_age_interval(source_age_interval, source_date_interval, target_date_interval)
        candidate_rows.append({
            'source_judgement_id': row['source_judgement_id'],
            'neutral_citation': row['neutral_citation'],
            'defendant_id': row['defendant_id'],
            'defendant_name': row['defendant_name'],
            'charge_nos': row['charge_nos'],
            'judgment_date': row['judgment_date'].isoformat(),
            'offence_date_range': date_interval_text(row['offence_date_interval']),
            'age_at_offence': format_age_output(row['age_at_offence_interval']),
            'age_at_sentencing': format_age_output(row['age_at_sentencing_interval']),
            'missing_field': 'age_at_offence',
            'inferred_age_at_offence': format_age_output(inferred_interval),
            'inferred_age_at_sentencing': None,
            'inferred_age_range': age_interval_text(inferred_interval),
        })

candidate_columns = [
    'source_judgement_id',
    'neutral_citation',
    'defendant_id',
    'defendant_name',
    'charge_nos',
    'judgment_date',
    'offence_date_range',
    'age_at_offence',
    'age_at_sentencing',
    'missing_field',
    'inferred_age_at_offence',
    'inferred_age_at_sentencing',
    'inferred_age_range',
]

age_inference_candidates = pd.DataFrame(candidate_rows, columns=candidate_columns)
if not age_inference_candidates.empty:
    age_inference_candidates = age_inference_candidates.sort_values(
        ['neutral_citation', 'defendant_id']
    ).reset_index(drop=True)

update_operations = []
for row in age_inference_candidates.to_dict('records'):
    if row['missing_field'] == 'age_at_offence':
        age_value = row['inferred_age_at_offence']
        source_text = (
            f"Inferred from {row['neutral_citation']} using age_at_sentencing, "
            f"offence_date {row['offence_date_range']} and judgment_date_time {row['judgment_date']}"
        )
    else:
        age_value = row['inferred_age_at_sentencing']
        source_text = (
            f"Inferred from {row['neutral_citation']} using age_at_offence, "
            f"offence_date {row['offence_date_range']} and judgment_date_time {row['judgment_date']}"
        )

    update_operations.append(
        UpdateOne(
            {
                '_id': to_object_id(row['source_judgement_id']),
                'defendants.defendants.defendant_id': row['defendant_id'],
            },
            {
                '$set': {
                    f"defendants.defendants.$[defendant].{row['missing_field']}": build_age_record(
                        age_value,
                        source_text,
                    ),
                }
            },
            array_filters=[
                {
                    'defendant.defendant_id': row['defendant_id'],
                }
            ],
        )
    )

update_result = None
if update_operations:
    update_result = verified_features.bulk_write(update_operations, ordered=False)

both_missing_cases = grouped_df[grouped_df['age_status'] == 'both_missing'].copy()
both_missing_cases['judgment_date'] = both_missing_cases['judgment_date'].apply(
    lambda value: value.isoformat() if value else None
)
both_missing_cases['offence_date_range'] = both_missing_cases['offence_date_interval'].apply(date_interval_text)
both_missing_cases = both_missing_cases[
    [
        'source_judgement_id',
        'neutral_citation',
        'defendant_id',
        'defendant_name',
        'charge_nos',
        'judgment_date',
        'offence_date_range',
    ]
]
if not both_missing_cases.empty:
    both_missing_cases = both_missing_cases.sort_values(['neutral_citation', 'defendant_id']).reset_index(drop=True)

both_missing_cases.to_excel('both_missing_cases.xlsx', index=False)
age_inference_candidates.to_excel('age_inference_candidates.xlsx', index=False)

age_inference_candidates


,source_judgement_id,neutral_citation,defendant_id,defendant_name,charge_nos,judgment_date,offence_date_range,age_at_offence,age_at_sentencing,missing_field,inferred_age_at_offence,inferred_age_at_sentencing,inferred_age_range
0,69ef1f4118fb37fe35c5fff0,[2021] HKCFI 1158,1,Rojas Montoya Juan Pablo,"[1, 2]",2021-03-25,2018-03-19 to 2019-03-06,None,32,age_at_offence,"[28, 30]",None,28 to 30
1,69f100aed477219d3bba3196,[2021] HKCFI 1393,1,Yeung Shek-kin,[1],2021-04-13,2019-06-04,None,57,age_at_offence,"[55, 56]",None,55 to 56
2,69b929cf469a9fdda698320e,[2021] HKCFI 1523,1,卓節巧,[1],2021-04-14,2018-11-23,None,50,age_at_offence,"[47, 48]",None,47 to 48
3,69f35a909b890e28b053f2ea,[2021] HKCFI 1815,1,Wong Tik-wai,[1],2021-06-03,2019-06-27 to 2019-06-28,None,26,age_at_offence,"[24, 25]",None,24 to 25
4,69c0f594e426ce80bdb0966c,[2021] HKCFI 1888,1,Lau Yuk-leong,[2],2021-05-21,2019-01-21,None,28,age_at_offence,"[25, 26]",None,25 to 26
...,...,...,...,...,...,...,...,...,...,...,...,...,...
320,69df10833814b54cb25b08fa,[2025] HKDC 786,1,BUSTOS ALEXIS DAVID,[1],2025-05-09,2024-02-20,None,31,age_at_offence,"[29, 30]",None,29 to 30
321,69df28d1a62297b67842330b,[2025] HKDC 81,1,WAN CHEUK HAANG,[1],2025-01-13,2023-05-19 to 2023-05-20,24,None,age_at_sentencing,None,"[25, 26]",25 to 26
322,69e74c8b8250411df7b12279,[2025] HKDC 849,1,羅兆輝,[1],2025-05-19,2024-05-10,None,44,age_at_offence,"[42, 43]",None,42 to 43
323,69f3705ba9ff3e065eec20ed,[2025] HKDC 939,1,LI PO LOI,[2],2025-06-03,2024-01-09,None,46,age_at_offence,"[44, 45]",None,44 to 45


In [14]:
both_missing_cases


,source_judgement_id,neutral_citation,defendant_id,defendant_name,charge_nos,judgment_date,offence_date_range
0,69d85f0b0bd0f7c2ddf2e2be,[2021] HKCFI 2554,1,Pootornphai Jakkrit,[1],2021-06-22,2020-01-19
1,69f30a4b6d98b48b26d72cd8,[2021] HKCFI 2711,1,黃濟群,[1],2020-07-22,2019-03-26
2,69bb7bd3ed8f9e045654e810,[2021] HKCFI 2772,1,"Tang Yiu-cheong, Jackie",[1],2021-08-26,2020-01-21
3,69e8a5ec6249ec766c124e16,[2021] HKCFI 3324,1,DAR ASIM TAHIR,"[1, 2, 3]",2021-08-31,2018-10-03 to 2018-10-04
4,69ccd951fead4f8406174b18,[2021] HKDC 82,1,CHAK WAI HO,"[2, 3]",2021-01-20,2019-02-24
5,69ccd90566d31eab4aee75bc,[2021] HKDC 951,1,林東明,[1],2021-07-09,2020-04-22
6,69ccb48fbd3bbc64c1f9c7c0,[2021] HKDC 970,1,Allaha Rakha Rahman,[1],2021-06-30,2020-10-05
7,69f06c997050fd549861b031,[2022] HKCFI 252,1,周耀忠,[1],2022-01-04,2020-01-13
8,69e8bbb773abf16984715749,[2022] HKCFI 3223,1,Ebhonun Akonjie Alex,"[1, 2]",2022-09-08,2020-06-07 to 2020-06-11
9,69f368ab9888868958b1d8bd,[2022] HKCFI 3859,2,JESSENIA CORREA NAVARRO,"[1, 2, 3]",2022-11-24,2018-03-02


In [15]:
print(f'candidate rows: {len(age_inference_candidates)}')
print(f'both missing rows: {len(both_missing_cases)}')
print(f'unique cases with missing ages: {age_inference_candidates["neutral_citation"].nunique()}')
print(f'unique cases with both ages missing: {both_missing_cases["neutral_citation"].nunique()}')
print(update_result)


candidate rows: 325
both missing rows: 46
unique cases with missing ages: 298
unique cases with both ages missing: 44
BulkWriteResult({'writeErrors': [], 'writeConcernErrors': [], 'nInserted': 0, 'nUpserted': 0, 'nMatched': 325, 'nModified': 325, 'nRemoved': 0, 'upserted': []}, acknowledged=True)
